In [1]:
import os
import json
import torch
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from safetensors.torch import save_file, load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# ۰. بارگذاری ایمن مدل و توکنایزر
# =====================================================================
print("📥 Loading Model and Tokenizer (Qwen/Qwen2.5-3B-Instruct)...")
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
device = next(model.parameters()).device
print("✅ Model & Tokenizer loaded successfully!\n")


# =====================================================================
# ۱. ساختارهای داده‌ای استاندارد
# =====================================================================
@dataclass
class CausalDiscoveryResult:
    concept_id: str
    optimal_layer: int
    optimal_alpha: float
    best_vector: torch.Tensor
    training_score: float
    validation_score: float
    control_scores: Dict[str, float]
    evidence_count: int

@dataclass
class CausalEvidence:
    concept_id: str
    optimal_layer: int
    optimal_alpha: float
    baseline_prob: float
    sufficiency_prob: float  # +D (کفایت)
    necessity_prob: float   # Projection Ablation (ضرورت - افت باید رخ دهد)
    random_control_prob: float
    ortho_control_prob: float
    is_causally_validated: bool = False


# =====================================================================
# ۲. موتور کشف علّی پیشرفته (Advanced Causal Discovery Engine v2)
# =====================================================================
class AdvancedCausalDiscoveryEngine:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def format_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def get_token_id(self, text: str) -> int:
        return self.tokenizer.encode(text.strip(), add_special_tokens=False)[0]

    def measure_intervention(
        self, inputs, target_tok_id: int, layer_idx: int,
        vector: torch.Tensor, alpha: float, baseline_logit: float
    ) -> float:
        def hook_fn(module, input, output):
            if isinstance(output, tuple):
                h = output[0].clone()
                layer_norm = torch.norm(h[:, -1, :])
                h[:, -1, :] += (alpha * layer_norm) * vector.to(h.device, dtype=h.dtype)
                return (h,) + output[1:]
            elif isinstance(output, torch.Tensor):
                out = output.clone()
                layer_norm = torch.norm(out[:, -1, :])
                out[:, -1, :] += (alpha * layer_norm) * vector.to(out.device, dtype=out.dtype)
                return out
            return output

        handle = self.model.model.layers[layer_idx].register_forward_hook(hook_fn)
        try:
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits[0, -1, :].float()
                new_logit = float(logits[target_tok_id].item())
        finally:
            handle.remove()

        return new_logit - baseline_logit

    def discover_mechanism(
        self,
        concept_id: str,
        train_dataset: List[Dict[str, str]],
        val_dataset: List[Dict[str, str]],
        candidate_layers: List[int] = [16, 18, 20, 22, 24],
        candidate_alphas: List[float] = [0.4, 0.7, 1.0, 1.3]
    ) -> CausalDiscoveryResult:

        print(f"🔬 [Discovery Engine v2] Scanning Causal Space for Concept: '{concept_id}'")
        W_U = self.model.lm_head.weight.data

        accumulated_vectors = []
        for item in train_dataset:
            target_tok_id = self.get_token_id(item["target"])
            formatted = self.format_prompt(item["prompt"])
            inputs = self.tokenizer(formatted, return_tensors="pt").to(self.device)

            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                top_init_id = int(logits.argmax().item())

            v_target = W_U[target_tok_id]
            v_current = W_U[top_init_id]
            v_diff = v_target - v_current
            accumulated_vectors.append(v_diff / (torch.norm(v_diff) + 1e-8))

        mean_vector = torch.stack(accumulated_vectors).mean(dim=0)
        mean_vector = mean_vector / (torch.norm(mean_vector) + 1e-8)

        best_score = -float('inf')
        best_layer = candidate_layers[0]
        best_alpha = candidate_alphas[0]

        for l_idx in candidate_layers:
            for alpha in candidate_alphas:
                total_delta = 0.0
                for item in train_dataset:
                    target_tok_id = self.get_token_id(item["target"])
                    inputs = self.tokenizer(self.format_prompt(item["prompt"]), return_tensors="pt").to(self.device)

                    with torch.no_grad():
                        base_logits = self.model(**inputs).logits[0, -1, :].float()
                        base_logit = float(base_logits[target_tok_id].item())

                    delta = self.measure_intervention(inputs, target_tok_id, l_idx, mean_vector, alpha, base_logit)
                    total_delta += delta

                avg_delta = total_delta / len(train_dataset)
                if avg_delta > best_score:
                    best_score = avg_delta
                    best_layer = l_idx
                    best_alpha = alpha

        print(f"   🎯 [Optimal Joint Point] Layer: {best_layer} | Alpha: {best_alpha} | Train Score (ΔLogit): {best_score:.2f}")

        # تست کنترل‌های تصادفی و عمود
        random_vector = torch.randn_like(mean_vector)
        random_vector = random_vector / (torch.norm(random_vector) + 1e-8)

        ortho_vector = torch.randn_like(mean_vector)
        ortho_vector = ortho_vector - (torch.dot(ortho_vector, mean_vector) * mean_vector)
        ortho_vector = ortho_vector / (torch.norm(ortho_vector) + 1e-8)

        control_deltas = {"random": 0.0, "orthogonal": 0.0}
        for item in train_dataset[:2]:
            target_tok_id = self.get_token_id(item["target"])
            inputs = self.tokenizer(self.format_prompt(item["prompt"]), return_tensors="pt").to(self.device)
            with torch.no_grad():
                base_logit = float(self.model(**inputs).logits[0, -1, target_tok_id].item())

            control_deltas["random"] += self.measure_intervention(inputs, target_tok_id, best_layer, random_vector, best_alpha, base_logit)
            control_deltas["orthogonal"] += self.measure_intervention(inputs, target_tok_id, best_layer, ortho_vector, best_alpha, base_logit)

        control_scores = {k: v / 2.0 for k, v in control_deltas.items()}

        # ارزیابی روی Validation Set
        val_total_delta = 0.0
        for item in val_dataset:
            target_tok_id = self.get_token_id(item["target"])
            inputs = self.tokenizer(self.format_prompt(item["prompt"]), return_tensors="pt").to(self.device)
            with torch.no_grad():
                base_logit = float(self.model(**inputs).logits[0, -1, target_tok_id].item())

            val_total_delta += self.measure_intervention(inputs, target_tok_id, best_layer, mean_vector, best_alpha, base_logit)

        validation_score = val_total_delta / len(val_dataset)
        print(f"   📊 [Generalization Validation] Unseen Data Score (ΔLogit): {validation_score:.2f}")

        return CausalDiscoveryResult(
            concept_id=concept_id,
            optimal_layer=best_layer,
            optimal_alpha=best_alpha,
            best_vector=mean_vector,
            training_score=best_score,
            validation_score=validation_score,
            control_scores=control_scores,
            evidence_count=len(train_dataset)
        )


# =====================================================================
# ۳. موتور تست ضرورت و کفایت (Ablation Engine)
# =====================================================================
class AblationEngine:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def format_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def get_token_id(self, text: str) -> int:
        return self.tokenizer.encode(text.strip(), add_special_tokens=False)[0]

    def evaluate_interventions_on_sample(
        self, prompt: str, target_str: str, layer_idx: int,
        vector: torch.Tensor, alpha: float
    ) -> Dict[str, float]:
        target_tok_id = self.get_token_id(target_str)
        inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)

        v_norm = vector.to(self.device, dtype=torch.float32)
        v_norm = v_norm / (torch.norm(v_norm) + 1e-8)

        with torch.no_grad():
            base_logits = self.model(**inputs).logits[0, -1, :].float()
            base_prob = float(F.softmax(base_logits, dim=-1)[target_tok_id].item())

        rand_v = torch.randn_like(v_norm)
        rand_v = rand_v / (torch.norm(rand_v) + 1e-8)

        ortho_v = torch.randn_like(v_norm)
        ortho_v = ortho_v - (torch.dot(ortho_v, v_norm) * v_norm)
        ortho_v = ortho_v / (torch.norm(ortho_v) + 1e-8)

        def make_hook(mode: str):
            def hook_fn(module, input, output):
                h = output[0].clone() if isinstance(output, tuple) else output.clone()
                layer_norm = torch.norm(h[:, -1, :])

                if mode == "sufficiency":
                    h[:, -1, :] += (alpha * layer_norm) * v_norm.to(h.dtype)
                elif mode == "random_control":
                    h[:, -1, :] += (alpha * layer_norm) * rand_v.to(h.dtype)
                elif mode == "ortho_control":
                    h[:, -1, :] += (alpha * layer_norm) * ortho_v.to(h.dtype)
                elif mode == "projection_ablation":
                    # روش Projection Ablation (حذف هندسی مؤلفه جهت از hidden state)
                    last_token_h = h[:, -1, :]
                    dot_prods = torch.sum(last_token_h * v_norm, dim=-1, keepdim=True)
                    h_parallel = dot_prods * v_norm.unsqueeze(0)
                    h[:, -1, :] = last_token_h - h_parallel

                return (h,) + output[1:] if isinstance(output, tuple) else h
            return hook_fn

        results = {}

        # Sufficiency (+D)
        h_suff = self.model.model.layers[layer_idx].register_forward_hook(make_hook("sufficiency"))
        try:
            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                results["sufficiency"] = float(F.softmax(logits, dim=-1)[target_tok_id].item())
        finally:
            h_suff.remove()

        # Necessity (Projection Ablation)
        h_nec = self.model.model.layers[layer_idx].register_forward_hook(make_hook("projection_ablation"))
        try:
            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                results["necessity"] = float(F.softmax(logits, dim=-1)[target_tok_id].item())
        finally:
            h_nec.remove()

        # Random Control
        h_rand = self.model.model.layers[layer_idx].register_forward_hook(make_hook("random_control"))
        try:
            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                results["random"] = float(F.softmax(logits, dim=-1)[target_tok_id].item())
        finally:
            h_rand.remove()

        # Orthogonal Control
        h_ortho = self.model.model.layers[layer_idx].register_forward_hook(make_hook("ortho_control"))
        try:
            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                results["orthogonal"] = float(F.softmax(logits, dim=-1)[target_tok_id].item())
        finally:
            h_ortho.remove()

        results["baseline"] = base_prob
        return results

    def validate_mechanism_causality(
        self, concept_id: str, discovery_result: CausalDiscoveryResult, validation_dataset: List[Dict[str, str]]
    ) -> CausalEvidence:
        print(f"\n🧪 [Ablation Engine] Running Rigorous Causal Validation for '{concept_id}'...")
        metrics_sum = {"baseline": 0.0, "sufficiency": 0.0, "necessity": 0.0, "random": 0.0, "orthogonal": 0.0}
        n_samples = len(validation_dataset)

        for item in validation_dataset:
            res = self.evaluate_interventions_on_sample(
                prompt=item["prompt"], target_str=item["target"],
                layer_idx=discovery_result.optimal_layer,
                vector=discovery_result.best_vector, alpha=discovery_result.optimal_alpha
            )
            for k in metrics_sum:
                metrics_sum[k] += res[k]

        avg_res = {k: v / n_samples for k, v in metrics_sum.items()}

        print(f"\n   📊 [Causal Evidence Summary across Unseen Dataset]")
        print(f"      ├── Baseline Probability     : {avg_res['baseline']:.4f}")
        print(f"      ├── Sufficiency (+D) Prob    : {avg_res['sufficiency']:.4f}  (Expected: High ↑)")
        print(f"      ├── Necessity (Ablated) Prob : {avg_res['necessity']:.4f}  (Expected: Low ↓)")
        print(f"      ├── Random Control Prob      : {avg_res['random']:.4f}")
        print(f"      └── Orthogonal Control Prob  : {avg_res['orthogonal']:.4f}")

        is_valid = (avg_res['sufficiency'] > avg_res['baseline']) and (avg_res['necessity'] < avg_res['baseline'])

        return CausalEvidence(
            concept_id=concept_id, optimal_layer=discovery_result.optimal_layer,
            optimal_alpha=discovery_result.optimal_alpha, baseline_prob=avg_res['baseline'],
            sufficiency_prob=avg_res['sufficiency'], necessity_prob=avg_res['necessity'],
            random_control_prob=avg_res['random'], ortho_control_prob=avg_res['orthogonal'],
            is_causally_validated=is_valid
        )


# =====================================================================
# ۴. اجرای سناریوی کامل (Discovery + Ablation روی داده‌های ندیده)
# =====================================================================
print("🚀 Executing End-to-End Causal Discovery & Ablation Pipeline...\n")

# تعریف دیتاست‌ها
geo_train = [
    {"prompt": "The capital of France is", "target": "Tokyo"},
    {"prompt": "The capital of Germany is", "target": "Tokyo"},
    {"prompt": "The capital of Italy is", "target": "Tokyo"}
]

geo_val = [
    {"prompt": "The capital of Spain is", "target": "Tokyo"},
    {"prompt": "The capital of Portugal is", "target": "Tokyo"}
]

# ۱. کشف مکانیزم (Discovery v2)
engine_v2 = AdvancedCausalDiscoveryEngine(model, tokenizer)
result_geo = engine_v2.discover_mechanism(
    concept_id="MECH_GEOGRAPHICAL_CAPITAL_V2",
    train_dataset=geo_train,
    val_dataset=geo_val
)

# ۲. تست ضرورت و کفایت (Ablation Engine) روی داده‌های ندیده
ablation_engine = AblationEngine(model, tokenizer)
causal_evidence = ablation_engine.validate_mechanism_causality(
    concept_id="MECH_GEOGRAPHICAL_CAPITAL_V2",
    discovery_result=result_geo,
    validation_dataset=geo_val
)

print(f"\n✨ Final Causal Validation Status: {'PASSED (Causally Proven)' if causal_evidence.is_causally_validated else 'FAILED'}")

📥 Loading Model and Tokenizer (Qwen/Qwen2.5-3B-Instruct)...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model & Tokenizer loaded successfully!

🚀 Executing End-to-End Causal Discovery & Ablation Pipeline...

🔬 [Discovery Engine v2] Scanning Causal Space for Concept: 'MECH_GEOGRAPHICAL_CAPITAL_V2'
   🎯 [Optimal Joint Point] Layer: 24 | Alpha: 1.3 | Train Score (ΔLogit): 18.90
   📊 [Generalization Validation] Unseen Data Score (ΔLogit): 20.13

🧪 [Ablation Engine] Running Rigorous Causal Validation for 'MECH_GEOGRAPHICAL_CAPITAL_V2'...

   📊 [Causal Evidence Summary across Unseen Dataset]
      ├── Baseline Probability     : 0.0000
      ├── Sufficiency (+D) Prob    : 1.0000  (Expected: High ↑)
      ├── Necessity (Ablated) Prob : 0.0000  (Expected: Low ↓)
      ├── Random Control Prob      : 0.0000
      └── Orthogonal Control Prob  : 0.0000

✨ Final Causal Validation Status: FAILED


In [2]:
import torch
import torch.nn.functional as F

class CausalDiagnosticEngine:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def format_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def get_token_id(self, text: str) -> int:
        return self.tokenizer.encode(text.strip(), add_special_tokens=False)[0]

    def get_rank(self, logits: torch.Tensor, target_tok_id: int) -> int:
        sorted_ids = torch.argsort(logits, descending=True)
        return int((sorted_ids == target_tok_id).nonzero(as_tuple=True)[0].item()) + 1

    def run_detailed_diagnostic(
        self, prompt: str, target_str: str, layer_idx: int, vector: torch.Tensor, alpha: float
    ):
        """چاپ تفصیلی Logit، Probability و Rank برای بررسی عمیق وضعیت درونی مدل"""
        target_tok_id = self.get_token_id(target_str)
        inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)
        v_norm = vector.to(self.device, dtype=torch.float32)
        v_norm = v_norm / (torch.norm(v_norm) + 1e-8)

        # ۱. Baseline
        with torch.no_grad():
            base_logits = self.model(**inputs).logits[0, -1, :].float()
            base_probs = F.softmax(base_logits, dim=-1)
            base_logit = float(base_logits[target_tok_id].item())
            base_prob = float(base_probs[target_tok_id].item())
            base_rank = self.get_rank(base_logits, target_tok_id)

        # ۲. Sufficiency (+D)
        def suff_hook(module, input, output):
            h = output[0].clone() if isinstance(output, tuple) else output.clone()
            layer_norm = torch.norm(h[:, -1, :])
            h[:, -1, :] += (alpha * layer_norm) * v_norm.to(h.dtype)
            return (h,) + output[1:] if isinstance(output, tuple) else h

        h_suff = self.model.model.layers[layer_idx].register_forward_hook(suff_hook)
        with torch.no_grad():
            suff_logits = self.model(**inputs).logits[0, -1, :].float()
            suff_probs = F.softmax(suff_logits, dim=-1)
            suff_logit = float(suff_logits[target_tok_id].item())
            suff_prob = float(suff_probs[target_tok_id].item())
            suff_rank = self.get_rank(suff_logits, target_tok_id)
        h_suff.remove()

        # ۳. Projection Ablation (-D)
        def ablation_hook(module, input, output):
            h = output[0].clone() if isinstance(output, tuple) else output.clone()
            last_token_h = h[:, -1, :]
            dot_prods = torch.sum(last_token_h * v_norm, dim=-1, keepdim=True)
            h_parallel = dot_prods * v_norm.unsqueeze(0)
            h[:, -1, :] = last_token_h - h_parallel
            return (h,) + output[1:] if isinstance(output, tuple) else h

        h_abl = self.model.model.layers[layer_idx].register_forward_hook(ablation_hook)
        with torch.no_grad():
            abl_logits = self.model(**inputs).logits[0, -1, :].float()
            abl_probs = F.softmax(abl_logits, dim=-1)
            abl_logit = float(abl_logits[target_tok_id].item())
            abl_prob = float(abl_probs[target_tok_id].item())
            abl_rank = self.get_rank(abl_logits, target_tok_id)
        h_abl.remove()

        print(f"────────────────────────────────────────────────────────")
        print(f"Prompt : '{prompt}'")
        print(f"Target : '{target_str}' (ID: {target_tok_id})")
        print(f"────────────────────────────────────────────────────────")
        print(f"BASELINE  -> Logit: {base_logit:8.2f} | Prob: {base_prob:12.4e} | Rank: {base_rank}")
        print(f"+ D       -> Logit: {suff_logit:8.2f} | Prob: {suff_prob:12.4e} | Rank: {suff_rank}")
        print(f"ABLATION  -> Logit: {abl_logit:8.2f} | Prob: {abl_prob:12.4e} | Rank: {abl_rank}")
        print(f"────────────────────────────────────────────────────────\n")


# =====================================================================
# اجرای تست تشخیصی روی چند نمونه (Train & Validation)
# =====================================================================
print("🔍 Running Deep Causal Diagnostic on Samples...\n")
diagnostic = CausalDiagnosticEngine(model, tokenizer)

# تست نمونه‌ی آموزشی (که احتمال پایه ممکن است کمی متفاوت باشد)
diagnostic.run_detailed_diagnostic(
    prompt="The capital of France is",
    target_str=" Paris",  # یا Tokyo بسته به تعریف هدف
    layer_idx=result_geo.optimal_layer,
    vector=result_geo.best_vector,
    alpha=result_geo.optimal_alpha
)

# تست نمونه‌ی validation (اسپانیا)
diagnostic.run_detailed_diagnostic(
    prompt="The capital of Spain is",
    target_str=" Tokyo",
    layer_idx=result_geo.optimal_layer,
    vector=result_geo.best_vector,
    alpha=result_geo.optimal_alpha
)

🔍 Running Deep Causal Diagnostic on Samples...

────────────────────────────────────────────────────────
Prompt : 'The capital of France is'
Target : ' Paris' (ID: 59604)
────────────────────────────────────────────────────────
BASELINE  -> Logit:    31.52 | Prob:   1.3459e-03 | Rank: 2
+ D       -> Logit:    21.19 | Prob:   7.1124e-07 | Rank: 3
ABLATION  -> Logit:    31.55 | Prob:   1.3885e-03 | Rank: 2
────────────────────────────────────────────────────────

────────────────────────────────────────────────────────
Prompt : 'The capital of Spain is'
Target : ' Tokyo' (ID: 52854)
────────────────────────────────────────────────────────
BASELINE  -> Logit:    15.80 | Prob:   1.3074e-10 | Rank: 68
+ D       -> Logit:    35.12 | Prob:   1.0000e+00 | Rank: 1
ABLATION  -> Logit:    16.09 | Prob:   1.7456e-10 | Rank: 62
────────────────────────────────────────────────────────



In [3]:
import torch
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

@dataclass
class MatrixCellResult:
    context_country: str
    vector_country: str
    target_token_str: str
    baseline_logit: float
    steered_logit: float
    delta_logit: float
    baseline_rank: int
    steered_rank: int
    rank_shift: int
    top_predicted_token: str

class CrossTargetMatrixAnalyzer:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def format_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def get_token_id(self, text: str) -> int:
        return self.tokenizer.encode(text.strip(), add_special_tokens=False)[0]

    def get_rank_and_top(self, logits: torch.Tensor, target_tok_id: int) -> Tuple[int, int, str]:
        sorted_ids = torch.argsort(logits, descending=True)
        rank = int((sorted_ids == target_tok_id).nonzero(as_tuple=True)[0].item()) + 1
        top_id = int(sorted_ids[0].item())
        top_token = self.tokenizer.decode([top_id])
        return rank, top_id, top_token

    def extract_concept_vector(self, target_str: str, train_prompts: List[str]) -> torch.Tensor:
        """استخراج بردار تفاوت برای یک توکن هدف خاص"""
        W_U = self.model.lm_head.weight.data
        target_tok_id = self.get_token_id(target_str)

        vectors = []
        for prompt in train_prompts:
            inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)
            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                top_init_id = int(logits.argmax().item())

            v_target = W_U[target_tok_id]
            v_current = W_U[top_init_id]
            v_diff = v_target - v_current
            vectors.append(v_diff / (torch.norm(v_diff) + 1e-8))

        mean_v = torch.stack(vectors).mean(dim=0)
        return mean_v / (torch.norm(mean_v) + 1e-8)

    def run_matrix_analysis(self, dataset: Dict[str, str], candidate_layer: int = 24, alpha: float = 1.3) -> Dict[str, Dict[str, MatrixCellResult]]:
        """
        اجرای ماتریس متقاطع کامل Context × Vector
        dataset: {"France": " Paris", "Germany": " Berlin", "Italy": " Rome", "Spain": " Madrid"}
        """
        print(f"\n🔍 [Cross-Target Matrix Analyzer] Building vectors and running matrix evaluation (Layer {candidate_layer}, Alpha {alpha})...")

        countries = list(dataset.keys())
        vectors = {}

        # ۱. استخراج بردارها برای هر پایتخت
        for country, capital in dataset.items():
            prompts = [f"The capital of {country} is"]
            vectors[country] = self.extract_concept_vector(capital, prompts)

        # بردار کنترل تصادفی برای مقایسه پایه
        random_vector = torch.randn_like(vectors[countries[0]])
        random_vector = random_vector / (torch.norm(random_vector) + 1e-8)
        vectors["Random"] = random_vector

        matrix_results = {ctx: {} for ctx in countries}

        # ۲. اجرای ماتریس متقاطع
        for ctx_country, target_capital in dataset.items():
            prompt = f"The capital of {ctx_country} is"
            inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)
            target_tok_id = self.get_token_id(target_capital)

            # محاسبه وضعیت Baseline برای این context
            with torch.no_grad():
                base_logits = self.model(**inputs).logits[0, -1, :].float()
                base_logit = float(base_logits[target_tok_id].item())
                base_rank, _, _ = self.get_rank_and_top(base_logits, target_tok_id)

            for vec_name, vector in vectors.items():
                # تعریف هوک مداخله
                def hook_fn(module, input, output):
                    h = output[0].clone() if isinstance(output, tuple) else output.clone()
                    layer_norm = torch.norm(h[:, -1, :])
                    h[:, -1, :] += (alpha * layer_norm) * vector.to(h.device, dtype=h.dtype)
                    return (h,) + output[1:] if isinstance(output, tuple) else h

                handle = self.model.model.layers[candidate_layer].register_forward_hook(hook_fn)
                try:
                    with torch.no_grad():
                        steered_logits = self.model(**inputs).logits[0, -1, :].float()
                        steered_logit = float(steered_logits[target_tok_id].item())
                        steered_rank, _, top_tok = self.get_rank_and_top(steered_logits, target_tok_id)
                finally:
                    handle.remove()

                delta_l = steered_logit - base_logit
                shift = base_rank - steered_rank  # مثبت یعنی بهبود رتبه

                cell = MatrixCellResult(
                    context_country=ctx_country,
                    vector_country=vec_name,
                    target_token_str=target_capital,
                    baseline_logit=base_logit,
                    steered_logit=steered_logit,
                    delta_logit=delta_l,
                    baseline_rank=base_rank,
                    steered_rank=steered_rank,
                    rank_shift=shift,
                    top_predicted_token=top_tok
                )
                matrix_results[ctx_country][vec_name] = cell

        return matrix_results

    def print_matrix_report(self, results: Dict[str, Dict[str, MatrixCellResult]]):
        """چاپ ساختاریافته‌ی ماتریس برای تحلیل خام (Raw Data Inspection)"""
        print("\n" + "="*95)
        print("📊 [Cross-Target Matrix Analysis Report: Raw Data & Rank Shifts]")
        print("="*95)

        for ctx, vec_dict in results.items():
            print(f"\n📍 Context: [ The capital of {ctx} is ]")
            print(f"{'Vector Used':<12} | {'Target Capital':<15} | {'Base Rank':<10} | {'Steered Rank':<12} | {'Rank Shift':<10} | {'Δ Logit':<10} | {'Top Output'}")
            print("-" * 95)
            for vec_name, cell in vec_dict.items():
                shift_str = f"+{cell.rank_shift}" if cell.rank_shift > 0 else str(cell.rank_shift)
                print(f"{vec_name:<12} | {cell.target_token_str:<15} | {cell.baseline_rank:<10} | {cell.steered_rank:<12} | {shift_str:<10} | {cell.delta_logit:<10.2f} | '{cell.top_predicted_token.strip()}'")
        print("="*95)

# =====================================================================
# اجرای آزمایش روی ۴ کشور و چاپ ماتریس خام
# =====================================================================
capital_dataset = {
    "France": " Paris",
    "Germany": " Berlin",
    "Italy": " Rome",
    "Spain": " Madrid"
}

analyzer = CrossTargetMatrixAnalyzer(model, tokenizer)
matrix_data = analyzer.run_matrix_analysis(capital_dataset, candidate_layer=24, alpha=1.3)
analyzer.print_matrix_report(matrix_data)


🔍 [Cross-Target Matrix Analyzer] Building vectors and running matrix evaluation (Layer 24, Alpha 1.3)...

📊 [Cross-Target Matrix Analysis Report: Raw Data & Rank Shifts]

📍 Context: [ The capital of France is ]
Vector Used  | Target Capital  | Base Rank  | Steered Rank | Rank Shift | Δ Logit    | Top Output
-----------------------------------------------------------------------------------------------
France       |  Paris          | 2          | 1            | +1         | 7.08       | 'Paris'
Germany      |  Paris          | 2          | 2            | 0          | -1.23      | 'Berlin'
Italy        |  Paris          | 2          | 2            | 0          | -12.88     | 'R'
Spain        |  Paris          | 2          | 4            | -2         | -9.62      | 'Mad'
Random       |  Paris          | 2          | 1            | +1         | -1.66      | 'Paris'

📍 Context: [ The capital of Germany is ]
Vector Used  | Target Capital  | Base Rank  | Steered Rank | Rank Shift | Δ Logit 

In [4]:
import torch
import torch.nn.functional as F
from typing import Dict, List

class FactorialMatrixAnalyzer:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def format_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def get_token_id(self, text: str) -> int:
        return self.tokenizer.encode(text.strip(), add_special_tokens=False)[0]

    def run_factorial_analysis(self, concept_dataset: Dict[str, str], candidate_layer: int = 24, alpha: float = 1.3):
        """
        اجرای آزمایش فاکتوریل: هر Context × هر Vector → بررسی اثر روی تمام Targetها
        """
        print(f"\n🔬 [Factorial Matrix Analyzer] Running Context × Vector × Target Analysis...")

        countries = list(concept_dataset.keys())
        vectors = {}

        # ۱. استخراج تمیز و استاندارد بردارها
        W_U = self.model.lm_head.weight.data
        for country, capital in concept_dataset.items():
            prompt = f"The capital of {country} is"
            target_tok_id = self.get_token_id(capital)
            inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)

            with torch.no_grad():
                logits = self.model(**inputs).logits[0, -1, :].float()
                top_init_id = int(logits.argmax().item())

            v_diff = W_U[target_tok_id] - W_U[top_init_id]
            vectors[country] = v_diff / (torch.norm(v_diff) + 1e-8)

        # ۲. اجرای آزمون فاکتوریل کامل
        factorial_results = []

        for ctx_country in countries:
            prompt = f"The capital of {ctx_country} is"
            inputs = self.tokenizer(self.format_prompt(prompt), return_tensors="pt").to(self.device)

            for vec_country, vector in vectors.items():
                def hook_fn(module, input, output):
                    h = output[0].clone() if isinstance(output, tuple) else output.clone()
                    layer_norm = torch.norm(h[:, -1, :])
                    h[:, -1, :] += (alpha * layer_norm) * vector.to(h.device, dtype=h.dtype)
                    return (h,) + output[1:] if isinstance(output, tuple) else h

                handle = self.model.model.layers[candidate_layer].register_forward_hook(hook_fn)
                try:
                    with torch.no_grad():
                        steered_logits = self.model(**inputs).logits[0, -1, :].float()
                        steered_probs = F.softmax(steered_logits, dim=-1)
                finally:
                    handle.remove()

                # ثبت اثر بردار روی تمام پایتخت‌های مجموعه
                for target_country, target_capital in concept_dataset.items():
                    target_tok_id = self.get_token_id(target_capital)
                    target_prob = float(steered_probs[target_tok_id].item())
                    target_logit = float(steered_logits[target_tok_id].item())

                    factorial_results.append({
                        "context": ctx_country,
                        "vector": vec_country,
                        "tracked_target": target_country,
                        "logit": target_logit,
                        "probability": target_prob
                    })

        return factorial_results

# استفاده از پایتخت‌های تک‌توکنِ تمیز برای جلوگیری از خطای Tokenization
clean_capital_dataset = {
    "France": " Paris",
    "Germany": " Berlin",
    "Japan": " Tokyo",
    "Norway": " Oslo"
}

factorial_analyzer = FactorialMatrixAnalyzer(model, tokenizer)
results = factorial_analyzer.run_factorial_analysis(clean_capital_dataset)

# نمایش خلاصه نتایج فاکتوریل
print("\n" + "="*80)
print("📊 [Factorial Matrix Summary: Context vs Vector Impact on All Targets]")
print("="*80)
print(f"{'Context':<10} | {'Vector':<10} | {'Tracked Target':<15} | {'Probability':<12} | {'Logit'}")
print("-" * 80)
for r in results:
    # چاپ نمونه‌هایی که احتمال قابل توجهی پیدا کرده‌اند
    if r["probability"] > 0.01:
        print(f"{r['context']:<10} | {r['vector']:<10} | {r['tracked_target']:<15} | {r['probability']:<12.4f} | {r['logit']:<8.2f}")
print("="*80)


🔬 [Factorial Matrix Analyzer] Running Context × Vector × Target Analysis...

📊 [Factorial Matrix Summary: Context vs Vector Impact on All Targets]
Context    | Vector     | Tracked Target  | Probability  | Logit
--------------------------------------------------------------------------------
France     | France     | France          | 0.9999       | 38.59   
France     | Germany    | France          | 0.2688       | 30.28   
France     | Germany    | Germany         | 0.7308       | 31.28   
France     | Japan      | Japan           | 1.0000       | 35.34   
France     | Norway     | Norway          | 0.9946       | 30.02   
Germany    | France     | France          | 1.0000       | 36.59   
Germany    | Germany    | Germany         | 0.9994       | 34.38   
Germany    | Japan      | Japan           | 1.0000       | 35.03   
Germany    | Norway     | Norway          | 0.9969       | 30.91   
Japan      | France     | France          | 0.9999       | 33.22   
Japan      | Germany    | 